# AI-Driven Data Analysis with PandasAI (CBORG Edition)

This notebook demonstrates how to use **PandasAI 3.0+** to perform natural language analysis across multiple materialized views in the `ca-biositing` project. 

It is configured to use the **CBORG API** (Gemini models) at LBL.

### Step 1: Setup and Environment

We start by loading our environment variables and initializing the LLM. This notebook uses the `ai-exploration` Pixi environment which contains the modern `pandasai` library.

**Note for PandasAI 3.0:** We use a custom LLM class to connect to the CBORG gateway and a custom patch for interactive Plotly charts.

In [ ]:
import os
import pandas as pd
import requests
import plotly.io as pio
import plotly.graph_objects as go
from dotenv import load_dotenv
from pandasai import Agent
from pandasai.llm.base import LLM
import pandasai.core.response.base as response_base
from sqlalchemy import create_engine
from IPython.display import HTML, display

# Set Plotly as default renderer for Jupyter
pio.renderers.default = 'notebook_connected'

# Load environment variables from .env
load_dotenv()

class CBORGLLM(LLM):
    """Custom LLM class for CBORG (OpenAI-compatible) gateway in PandasAI 3.0"""
    def __init__(self, api_token, api_base="https://api.cborg.lbl.gov/v1", model="gemini-2.0-flash"):
        super().__init__()
        self.api_token = api_token
        self.api_base = api_base
        self.model = model

    def call(self, instruction, context=None):
        prompt = instruction.to_string()
        headers = {
            "Authorization": f"Bearer {self.api_token}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0
        }
        response = requests.post(f"{self.api_base}/chat/completions", headers=headers, json=payload)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]

    @property
    def type(self):
        return "cborg"

# --- GLOBAL PATCH FOR INTERACTIVE RENDERING ---
def patched_repr_html(self):
    """Ensure any response containing an HTML path or Plotly object renders in the notebook"""
    try:
        if hasattr(self.value, 'to_html'):
            return self.value.to_html(include_plotlyjs='cdn', full_html=False)
        
        if isinstance(self.value, str) and self.value.endswith('.html'):
            fname = os.path.basename(self.value)
            paths_to_try = [
                self.value,
                os.path.join(os.getcwd(), self.value),
                os.path.abspath(os.path.join(os.getcwd(), "..", "..", self.value)),
                os.path.abspath(os.path.join(os.getcwd(), "..", "..", "exports", "charts", fname))
            ]
            for path in paths_to_try:
                if os.path.exists(path):
                    with open(path, 'r') as f:
                        return f.read()
    except Exception as e:
        return f"<p style='color:red'>Rendering Error: {e}</p>"
    return None

response_base.BaseResponse._repr_html_ = patched_repr_html
print("PandasAI patched globally for interactive Plotly rendering.")

api_key = os.getenv("CBORG_API_KEY")
api_url = os.getenv("CBORG_API_URL", "https://api.cborg.lbl.gov/v1")
model_name = os.getenv("CBORG_MODEL", "gemini-2.0-flash")

if not api_key:
    print("WARNING: CBORG_API_KEY not found in environment variables. Please check your .env file.")

# Initialize the LLM via CBORG gateway
llm = CBORGLLM(api_token=api_key, api_base=api_url, model=model_name)

print(f"LLM initialized with model: {model_name} at {api_url}")

### Step 2: Database Connection

We connect to the database to fetch the materialized views. By default, this connects to the local development database.

In [ ]:
DB_USER = os.getenv("DB_USER", "biocirv_user")
DB_PASS = os.getenv("DB_PASSWORD", "biocirv_dev_password")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "biocirv_db")
DB_SCHEMA = os.getenv("DB_SCHEMA", "ca_biositing")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

try:
    engine = create_engine(DATABASE_URL)
    with engine.connect() as conn:
        print(f"Connected to database: {DB_NAME} on {DB_HOST}")
except Exception as e:
    print(f"Error connecting to database: {e}")

### Step 3: Load DataFrames

Fetch the materialized views into Pandas DataFrames for analysis.

In [ ]:
views = ["analysis_data_view", "usda_census_view", "analysis_average_view"]
loaded_dfs = {}
for v in views:
    try:
        loaded_dfs[v] = pd.read_sql(f"SELECT * FROM {DB_SCHEMA}.{v}", engine)
        print(f"- Loaded {v}: {len(loaded_dfs[v])} rows")
    except Exception as e:
        print(f"- Skipping {v}: {e}")

df_analysis = loaded_dfs.get('analysis_data_view', pd.DataFrame())
df_usda = loaded_dfs.get('usda_census_view', pd.DataFrame())

### Step 4: AI Analysis with Agent

Now we can query the data using natural language. PandasAI will generate and execute Python code to answer your questions.

**Interactive Charts:** We applied a global patch to ensure interactive Plotly visualizations are displayed.

In [ ]:
# Wrap the dataframes in an Agent
agent = Agent(
    list(loaded_dfs.values()), 
    config={
        "llm": llm,
        "verbose": True
    }
)

print(f"Agent initialized with {len(loaded_dfs)} datasets.")

### Step 5: Visualizations

You can also ask the AI to generate interactive charts.

In [ ]:
agent.chat("Show me an interactive plotly bar chart of the top 5 resources by volume from the analysis data")